# 04 图像分割与形态学

## 概述

图像分割是将图像划分为多个有意义的区域或对象的过程。形态学操作则是对二值图像进行处理，改变区域形状的一系列操作。本模块将从基础阈值分割开始，逐步深入到分水岭算法、GrabCut 等高级分割方法，并结合形态学操作解决实际问题。

学习目标：
1. 掌握全局阈值、Otsu 自动阈值和自适应阈值
2. 理解并实现分水岭分割算法
3. 使用 GrabCut 进行交互式前景提取
4. 掌握形态学操作：腐蚀、膨胀、开运算、闭运算
5. 实现硬币计数和细胞计数等实际应用

In [ ]:
# ===== 环境准备 =====
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color, filters, morphology, measure, segmentation
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.morphology import disk, square
from scipy import ndimage as ndi
import cv2

from utils.image_utils import (load_image, display_image, display_images,
                                compare_images, ensure_uint8, ensure_float)
from utils.sample_data import checkerboard
from utils.visualization import plot_histogram

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print('环境准备完毕！')

## 全局阈值 vs Otsu 自动阈值

阈值分割是最简单的图像分割方法。核心思想是：选择一个阈值 T，将像素分为两类（前景和背景）。

### 全局阈值（Global Threshold）
人为指定一个固定阈值（如 128），所有像素大于该值的归为前景，否则归为背景。缺点是对于光照不均匀的图像效果很差。

### Otsu 大津法
Otsu 方法自动寻找最佳阈值，使得前景和背景之间的类间方差最大化。它遍历所有可能的阈值，选择使得"类间方差"最大的那个。适用于灰度直方图具有双峰特征的图像。

In [ ]:
# ===== 全局阈值与 Otsu 对比 =====

# 加载测试图片
img = data.camera()
print(f'图片尺寸: {img.shape}, 值范围: [{img.min()}, {img.max()}]')

# 全局阈值 128
thresh_global = img > 128

# Otsu 自动阈值
otsu_thresh_val = filters.threshold_otsu(img)
thresh_otsu = img > otsu_thresh_val
print(f'Otsu 自动计算阈值: {otsu_thresh_val:.1f}')

# 并排对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('原始图像')
axes[0].axis('off')

axes[1].imshow(thresh_global, cmap='gray')
axes[1].set_title(f'全局阈值 (T=128)')
axes[1].axis('off')

axes[2].imshow(thresh_otsu, cmap='gray')
axes[2].set_title(f'Otsu 自动阈值 (T={otsu_thresh_val:.1f})')
axes[2].axis('off')

plt.suptitle('全局阈值 vs Otsu 自动阈值', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 直方图 + 阈值线标注 =====

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 直方图
axes[0].hist(img.ravel(), bins=256, range=(0, 255), color='gray', alpha=0.7)
axes[0].axvline(x=128, color='red', linestyle='--', linewidth=2, label='全局阈值=128')
axes[0].axvline(x=otsu_thresh_val, color='blue', linestyle='--', linewidth=2,
                label=f'Otsu={otsu_thresh_val:.1f}')
axes[0].set_xlabel('像素值')
axes[0].set_ylabel('频率')
axes[0].set_title('灰度直方图与阈值分割线')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 放大阈值附近区域
axes[1].hist(img.ravel(), bins=256, range=(0, 255), color='gray', alpha=0.7)
axes[1].axvline(x=128, color='red', linestyle='--', linewidth=2, label='全局阈值=128')
axes[1].axvline(x=otsu_thresh_val, color='blue', linestyle='--', linewidth=2,
                label=f'Otsu={otsu_thresh_val:.1f}')
axes[1].set_xlim(50, 200)
axes[1].set_xlabel('像素值')
axes[1].set_ylabel('频率')
axes[1].set_title('直方图局部放大（阈值区域）')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('阈值分割的直方图视角', fontsize=14)
plt.tight_layout()
plt.show()

print('观察要点：')
print('- Otsu 阈值正好位于两个峰之间的谷底')
print('- 这最大化了前景和背景的分离度')
print('- 如果直方图不是双峰的，Otsu 的效果会变差')

## 自适应阈值

当图像光照不均匀时，全局阈值无法很好地分割。自适应阈值方法对图像中的每个小区域分别计算阈值，从而适应局部的亮度变化。

两种常用的自适应阈值方法：
- **自适应均值**（Adaptive Mean）：以邻域像素的均值作为阈值
- **自适应高斯**（Adaptive Gaussian）：以邻域像素的高斯加权均值作为阈值

这两种方法通过 `blockSize` 控制邻域大小，通过 `C`（常数）微调阈值。

In [ ]:
# ===== 自适应阈值对比 =====

# 创建一张光照不均匀的模拟图像
h, w = 256, 256
x, y = np.meshgrid(np.arange(w), np.arange(h))
# 渐变背景 + 文字区域
img_gradient = ((x / w * 200) + 30).astype(np.uint8)
# 添加一些"文字"方块
img_gradient[60:80, 40:200] = 200
img_gradient[120:140, 30:210] = 200
img_gradient[180:200, 50:180] = 200
img_gradient = img_gradient.astype(np.uint8)

# 全局阈值（效果差）
_, thresh_global = cv2.threshold(img_gradient, 128, 255, cv2.THRESH_BINARY)

# 自适应均值阈值
thresh_mean = cv2.adaptiveThreshold(img_gradient, 255,
                                     cv2.ADAPTIVE_THRESH_MEAN_C,
                                     cv2.THRESH_BINARY, 31, 10)

# 自适应高斯阈值
thresh_gaussian = cv2.adaptiveThreshold(img_gradient, 255,
                                         cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                         cv2.THRESH_BINARY, 31, 10)

# 四宫格对比
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(img_gradient, cmap='gray')
axes[0, 0].set_title('原始图像（左亮右暗）')
axes[0, 0].axis('off')

axes[0, 1].imshow(thresh_global, cmap='gray')
axes[0, 1].set_title('全局阈值 (T=128)\n右侧文字丢失')
axes[0, 1].axis('off')

axes[1, 0].imshow(thresh_mean, cmap='gray')
axes[1, 0].set_title('自适应均值阈值 (blockSize=31)')
axes[1, 0].axis('off')

axes[1, 1].imshow(thresh_gaussian, cmap='gray')
axes[1, 1].set_title('自适应高斯阈值 (blockSize=31)')
axes[1, 1].axis('off')

plt.suptitle('自适应阈值 vs 全局阈值（光照不均匀场景）', fontsize=14)
plt.tight_layout()
plt.show()

print('观察要点：')
print('- 全局阈值在右侧暗区域丢失了"文字"块')
print('- 自适应方法对每个小区域独立计算阈值，不受整体光照影响')
print('- 自适应高斯阈值比均值阈值更加平滑')

## 分水岭分割（Watershed）

分水岭算法将灰度图像看作地形图，将像素值看作海拔高度。算法模拟"注水"过程：

1. 在局部最小值处打孔
2. 从最低处开始注水
3. 当不同山谷的水即将汇合时，筑起大坝（分割线）
4. 最终的大坝就是分割边界

关键步骤：
- **距离变换**（Distance Transform）：计算每个像素到最近背景像素的距离
- **寻找标记**（Markers）：找到距离变换图的局部最大值作为"种子"
- **分水岭**：以标记为起点执行分水岭算法

适用于分离相互接触的圆形物体（如硬币、细胞）。

In [ ]:
# ===== 分水岭分割完整演示 =====

# 生成重叠的圆形图像
def generate_overlapping_circles(size=300):
    """生成模拟的重叠圆形对象"""
    image = np.zeros((size, size), dtype=np.uint8)
    circles = [
        (size//3, size//3, 40),
        (size//3 + 20, size//3 + 25, 38),
        (2*size//3, size//2, 42),
        (2*size//3 + 18, size//2 - 20, 35),
        (size//2, 2*size//3, 45),
        (size//2 + 22, 2*size//3 + 15, 40),
        (size//4, 3*size//4, 36),
        (3*size//4, size//4, 38),
    ]
    for cy, cx, r in circles:
        cv2.circle(image, (cx, cy), r, 255, -1)
    return image


img_circles = generate_overlapping_circles(300)
print(f'重叠圆形图像尺寸: {img_circles.shape}')

# 步骤1: 距离变换
dist_transform = cv2.distanceTransform(img_circles, cv2.DIST_L2, 5)
dist_norm = cv2.normalize(dist_transform, None, 0, 1.0, cv2.NORM_MINMAX)

# 步骤2: 找到局部最大值作为标记
# 二值化距离图
_, sure_fg = cv2.threshold(dist_transform, 0.3 * dist_transform.max(), 255, 0)
sure_fg = np.uint8(sure_fg)

# 步骤3: 获取未知区域
kernel = np.ones((3, 3), np.uint8)
sure_bg = cv2.dilate(img_circles, kernel, iterations=3)
unknown = cv2.subtract(sure_bg, sure_fg)

# 步骤4: 标记连通区域
_, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1
markers[unknown == 255] = 0

# 步骤5: 应用分水岭
markers = cv2.watershed(cv2.cvtColor(img_circles, cv2.COLOR_GRAY2BGR), markers)

# 可视化
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(img_circles, cmap='gray')
axes[0, 0].set_title('原始重叠圆形')
axes[0, 0].axis('off')

axes[0, 1].imshow(dist_norm, cmap='jet')
axes[0, 1].set_title('距离变换')
axes[0, 1].axis('off')

axes[0, 2].imshow(sure_fg, cmap='gray')
axes[0, 2].set_title('确定前景 (种子点)')
axes[0, 2].axis('off')

axes[1, 0].imshow(sure_bg, cmap='gray')
axes[1, 0].set_title('确定背景')
axes[1, 0].axis('off')

axes[1, 1].imshow(img_circles, cmap='gray')
# 绘制分割边界（markers == -1）
axes[1, 1].contour(markers == -1, colors='red', linewidths=1.5)
axes[1, 1].set_title('分水岭分割结果\n(红线 = 分割边界)')
axes[1, 1].axis('off')

# 彩色标记每个分割区域
axes[1, 2].imshow(markers, cmap='tab20')
axes[1, 2].set_title(f'区域标记\n(共 {markers.max() - 1} 个区域)')
axes[1, 2].axis('off')

plt.suptitle('分水岭分割完整流程', fontsize=14)
plt.tight_layout()
plt.show()

print(f'检测到的区域数: {markers.max() - 1}')

## GrabCut 前景提取

GrabCut 是一种交互式前景提取算法，基于图割（Graph Cut）理论。用户只需要用一个矩形框大致标出前景区域，算法会自动分离前景和背景。

核心原理：
1. 用高斯混合模型（GMM）对前景和背景的颜色分布建模
2. 构建像素之间的图结构（像素是节点，相邻像素之间有边）
3. 通过最小割（Min-Cut）算法找到最优的前景/背景分割
4. 迭代优化 GMM 参数和分割结果

这是 Photoshop 等图像编辑软件中"快速选择"和"魔术棒"的底层算法之一。

In [ ]:
# ===== GrabCut 分割演示 =====

# 使用合成图像：白色圆形前景 + 纹理背景
def make_grabcut_test_image(size=300):
    """生成 GrabCut 测试图像：前景物体在纹理背景上"""
    np.random.seed(42)
    # 背景纹理
    bg = np.random.randint(40, 120, (size, size, 3), dtype=np.uint8)
    # 添加条纹纹理
    for i in range(0, size, 8):
        bg[:, i:i+4, :] = np.clip(bg[:, i:i+4, :] + 30, 0, 255)

    # 前景：彩色圆形
    fg_color = np.array([220, 80, 60], dtype=np.uint8)
    y, x = np.ogrid[:size, :size]
    mask = ((x - size//2 + 10)**2 + (y - size//2 + 10)**2) <= (size//4)**2
    bg[mask] = fg_color

    # 前景中的小圆
    mask2 = ((x - size//2 - 30)**2 + (y - size//2 + 30)**2) <= (size//8)**2
    bg[mask2] = np.array([60, 180, 80], dtype=np.uint8)

    return bg, mask | mask2


img_grab, gt_mask = make_grabcut_test_image(300)
h_g, w_g = img_grab.shape[:2]

# GrabCut 需要初始化 mask
mask = np.zeros((h_g, w_g), np.uint8)
bgd_model = np.zeros((1, 65), np.float64)
fgd_model = np.zeros((1, 65), np.float64)

# 定义前景矩形（大致框住前景对象）
rect = (w_g//4, h_g//4, w_g//2 + 20, h_g//2 + 20)

# 执行 GrabCut（5 次迭代）
cv2.grabCut(img_grab, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)

# 提取前景
mask_fg = np.where((mask == 1) | (mask == 3), 1, 0).astype(np.uint8)
fg_extracted = img_grab * mask_fg[:, :, np.newaxis]

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_grab)
# 画矩形
from matplotlib.patches import Rectangle
rect_patch = Rectangle((rect[0], rect[1]), rect[2], rect[3],
                        fill=False, edgecolor='red', linewidth=2)
axes[0].add_patch(rect_patch)
axes[0].set_title('输入图像 + 前景矩形')
axes[0].axis('off')

axes[1].imshow(mask_fg, cmap='gray')
axes[1].set_title('GrabCut 前景掩膜')
axes[1].axis('off')

axes[2].imshow(fg_extracted)
axes[2].set_title('提取的前景')
axes[2].axis('off')

plt.suptitle('GrabCut 前景分割', fontsize=14)
plt.tight_layout()
plt.show()

## 形态学操作

形态学操作是对二值图像进行处理的基本工具。它们使用一个称为"结构元素（Structuring Element）"的小形状在图像上滑动，根据与邻域的关系修改像素值。

### 四种基本操作

| 操作 | 效果 | 公式 | 用途 |
|------|------|------|------|
| **腐蚀 (Erosion)** | 缩小白色区域 | 结构元素内全白才保留白 | 去除细小噪点、分离粘连物体 |
| **膨胀 (Dilation)** | 扩大白色区域 | 结构元素内任一白则输出白 | 填补空洞、连接断裂区域 |
| **开运算 (Opening)** | 先腐蚀后膨胀 | Open = Dilate(Erode(img)) | 去除小噪点、平滑边界 |
| **闭运算 (Closing)** | 先膨胀后腐蚀 | Close = Erode(Dilate(img)) | 填补小空洞、闭合缺口 |

结构元素可以用 cv2.getStructuringElement 创建：矩形 (MORPH_RECT)、椭圆 (MORPH_ELLIPSE)、十字形 (MORPH_CROSS)。

In [ ]:
# ===== 形态学操作完整演示 =====

# 创建二值测试图像（带噪点和空洞）
def create_morph_test_image(size=200):
    """创建带噪点和空洞的二值测试图像"""
    img = np.zeros((size, size), dtype=np.uint8)

    # 主矩形
    cv2.rectangle(img, (40, 40), (160, 160), 255, -1)

    # 添加随机白噪点（模拟噪点）
    np.random.seed(42)
    noise = np.random.random((size, size)) > 0.97
    img[noise] = 255

    # 添加随机黑点（模拟空洞）
    holes = np.random.random((size, size)) > 0.95
    img[50:150, 50:150][holes[50:150, 50:150]] = 0

    return img


img_binary = create_morph_test_image(200)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

# 执行四种形态学操作
eroded = cv2.erode(img_binary, kernel, iterations=1)
dilated = cv2.dilate(img_binary, kernel, iterations=1)
opened = cv2.morphologyEx(img_binary, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(img_binary, cv2.MORPH_CLOSE, kernel)

# 2x2 网格展示
results = [
    ('腐蚀 (Erosion)', eroded, '缩小白色区域，去除噪点'),
    ('膨胀 (Dilation)', dilated, '扩大白色区域，填补空洞'),
    ('开运算 (Opening)', opened, '先腐蚀后膨胀，去噪点保形状'),
    ('闭运算 (Closing)', closed, '先膨胀后腐蚀，填空洞保形状'),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

axes[0, 0].imshow(img_binary, cmap='gray')
axes[0, 0].set_title('原始二值图像\n(含噪点和空洞)')
axes[0, 0].axis('off')

for idx, (name, res, desc) in enumerate(results):
    ax = axes[(idx + 1) // 3, (idx + 1) % 3]
    ax.imshow(res, cmap='gray')
    ax.set_title(f'{name}\n{desc}', fontsize=10)
    ax.axis('off')

plt.suptitle('形态学操作对比（结构元素: 5x5 椭圆）', fontsize=14)
plt.tight_layout()
plt.show()

## 硬币计数应用

这是一个经典的分割+计数应用。流程如下：

1. 生成模拟的硬币图像（多个圆形在纹理背景上）
2. 阈值分割得到二值图
3. 距离变换找到每个硬币的中心区域
4. 分水岭分割分离粘连的硬币
5. 查找轮廓并计数
6. 在原图上标注每个硬币的编号

这个流程同样适用于工业零件计数、细胞计数等场景。

In [ ]:
# ===== 硬币计数完整流程 =====

def generate_coin_image(size=400, num_coins=12):
    """生成模拟硬币图像"""
    np.random.seed(42)
    # 纹理背景
    bg = np.random.randint(60, 130, (size, size), dtype=np.uint8)
    # 添加纹理
    for i in range(0, size, 10):
        bg[:, i:i+3] = np.clip(bg[:, i:i+3] + 15, 0, 255)

    # 放置硬币
    image = bg.copy()
    coins = []
    attempts = 0
    while len(coins) < num_coins and attempts < 200:
        cx = np.random.randint(60, size - 60)
        cy = np.random.randint(60, size - 60)
        r = np.random.randint(25, 42)

        # 检查与已放置硬币的重叠
        overlap = False
        for (pcx, pcy, pr) in coins:
            if np.sqrt((cx - pcx)**2 + (cy - pcy)**2) < (r + pr) * 0.85:
                overlap = True
                break
        if not overlap:
            cv2.circle(image, (cx, cy), r, 220, -1)
            coins.append((cx, cy, r))
        attempts += 1

    # 添加噪声
    noise = np.random.randint(-15, 15, (size, size))
    image = np.clip(image.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    return image, coins


# 生成硬币图像
coin_img, true_coins = generate_coin_image(400, 12)
print(f'生成硬币数（真实值）: {len(true_coins)}')

# ---- 硬币检测计数流程 ----
# 1. 高斯模糊
blurred = cv2.GaussianBlur(coin_img, (7, 7), 0)

# 2. Otsu 阈值
_, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# 3. 形态学开运算去噪
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
binary_clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)

# 4. 距离变换
dist = cv2.distanceTransform(binary_clean, cv2.DIST_L2, 5)

# 5. 找局部最大值
_, sure_fg = cv2.threshold(dist, 0.35 * dist.max(), 255, 0)
sure_fg = np.uint8(sure_fg)

# 6. 分水岭标记
_, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1
sure_bg = cv2.dilate(binary_clean, kernel, iterations=3)
unknown = cv2.subtract(sure_bg, sure_fg)
markers[unknown == 255] = 0
markers = cv2.watershed(cv2.cvtColor(coin_img, cv2.COLOR_GRAY2BGR), markers)

# 7. 统计硬币
num_detected = markers.max() - 1
print(f'检测到的硬币数: {num_detected}')

# 可视化
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(coin_img, cmap='gray')
axes[0, 0].set_title(f'原始硬币图像\n(真实数量: {len(true_coins)})')
axes[0, 0].axis('off')

axes[0, 1].imshow(binary, cmap='gray')
axes[0, 1].set_title('Otsu 阈值分割')
axes[0, 1].axis('off')

axes[0, 2].imshow(dist, cmap='jet')
axes[0, 2].set_title('距离变换')
axes[0, 2].axis('off')

axes[1, 0].imshow(sure_fg, cmap='gray')
axes[1, 0].set_title('确定前景 (种子)')
axes[1, 0].axis('off')

# 标记结果
coin_labeled = coin_img.copy()
axes[1, 1].imshow(coin_labeled, cmap='gray')
# 画分割边界
axes[1, 1].contour(markers == -1, colors='red', linewidths=1.5)
axes[1, 1].set_title(f'分水岭分割\n(检测到 {num_detected} 个硬币)')
axes[1, 1].axis('off')

axes[1, 2].imshow(markers, cmap='tab20')
axes[1, 2].set_title('区域着色')
axes[1, 2].axis('off')

plt.suptitle('硬币计数完整流程', fontsize=14)
plt.tight_layout()
plt.show()

## 练习与实践

通过以下练习加深对图像分割和形态学操作的理解。

In [ ]:
# 练习1: 修改形态学操作的结构元素大小，观察对结果的影响
# TODO: 完成以下练习

img_test = create_morph_test_image(200)

# 不同的结构元素大小
kernel_sizes = [3, 5, 7, 9]
kernels = [cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)) for k in kernel_sizes]

fig, axes = plt.subplots(4, 4, figsize=(16, 16))

for col, (k_size, kernel) in enumerate(zip(kernel_sizes, kernels)):
    # 开运算
    opened = cv2.morphologyEx(img_test, cv2.MORPH_OPEN, kernel)
    # 闭运算
    closed = cv2.morphologyEx(img_test, cv2.MORPH_CLOSE, kernel)
    # 腐蚀
    eroded = cv2.erode(img_test, kernel)
    # 膨胀
    dilated = cv2.dilate(img_test, kernel)

    results = [eroded, dilated, opened, closed]
    titles = ['腐蚀', '膨胀', '开运算', '闭运算']

    for row in range(4):
        ax = axes[col, row]
        ax.imshow(results[row], cmap='gray')
        ax.set_title(f'{titles[row]} (核={k_size})', fontsize=9)
        ax.axis('off')

plt.suptitle('不同结构元素大小对形态学操作的影响', fontsize=14)
plt.tight_layout()
plt.show()

print('观察要点：')
print('- 核越大，腐蚀/膨胀的效果越强')
print('- 过大的核会导致小物体完全消失')
print('- 选择合适的核大小取决于要去除的噪点/空洞尺寸')

In [ ]:
# 练习2: 对分割结果进行后处理——使用形态学闭运算填补空洞，开运算去除噪点
# TODO: 完成以下练习

# 创建一个带缺陷的分割结果
def create_defective_segmentation(size=200):
    """创建有孔洞和噪点的分割掩膜"""
    mask = np.zeros((size, size), dtype=np.uint8)
    # 主圆
    cv2.circle(mask, (size//2, size//2), 70, 255, -1)
    # 矩形
    cv2.rectangle(mask, (30, 100), (70, 170), 255, -1)

    # 添加噪点（白色小点）
    np.random.seed(123)
    for _ in range(80):
        nx, ny = np.random.randint(0, size), np.random.randint(0, size)
        cv2.circle(mask, (nx, ny), np.random.randint(1, 4), 255, -1)

    # 添加空洞（黑色小洞）
    for _ in range(15):
        hx = np.random.randint(size//2 - 60, size//2 + 60)
        hy = np.random.randint(size//2 - 60, size//2 + 60)
        cv2.circle(mask, (hx, hy), np.random.randint(2, 7), 0, -1)

    return mask


mask_defective = create_defective_segmentation(200)

# TODO: 使用形态学操作进行后处理
# 第一步：闭运算填补空洞
kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
mask_filled = cv2.morphologyEx(mask_defective, cv2.MORPH_CLOSE, kernel_close)

# 第二步：开运算去除噪点
kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
mask_clean = cv2.morphologyEx(mask_filled, cv2.MORPH_OPEN, kernel_open)

# 可视化前中后
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(mask_defective, cmap='gray')
axes[0].set_title('原始分割掩膜\n(含噪点和空洞)')
axes[0].axis('off')

axes[1].imshow(mask_filled, cmap='gray')
axes[1].set_title('闭运算后\n(空洞被填补)')
axes[1].axis('off')

axes[2].imshow(mask_clean, cmap='gray')
axes[2].set_title('再开运算\n(噪点被去除)')
axes[2].axis('off')

plt.suptitle('形态学后处理：填补空洞 + 去除噪点', fontsize=14)
plt.tight_layout()
plt.show()

# 统计改善
def count_holes(mask):
    """估算空洞数量"""
    inverted = 255 - mask
    _, labels = cv2.connectedComponents(inverted)
    # 减去背景标签
    return labels.max() - 1

print('后处理效果统计:')
print(f'  处理前空洞数: {count_holes(mask_defective)}')
print(f'  闭运算后空洞数: {count_holes(mask_filled)}')
print(f'  最终空洞数: {count_holes(mask_clean)}')

In [ ]:
# ===== 交互式阈值与形态学参数探索 =====

from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, fixed
from skimage.filters import threshold_otsu, threshold_local

img_coins_explore = data.coins()

def explore_threshold(threshold=128, method='global', block_size=51):
    """交互式探索不同阈值和分割方法"""
    if method == 'otsu':
        thresh_val = threshold_otsu(img_coins_explore)
        binary = img_coins_explore > thresh_val
        title = f'Otsu 自动阈值 = {thresh_val:.0f}'
    elif method == 'adaptive':
        thresh_local = threshold_local(
            img_coins_explore, block_size=block_size,
            method='gaussian', offset=threshold - 128
        )
        binary = img_coins_explore > thresh_local
        title = f'自适应阈值 (block={block_size}, offset={threshold-128})'
    else:
        binary = img_coins_explore > threshold
        title = f'固定阈值 = {threshold}'

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(img_coins_explore, cmap='gray')
    axes[0].set_title('原始图片')
    axes[0].axis('off')
    axes[1].hist(img_coins_explore.ravel(), bins=256, range=(0, 255), color='gray', alpha=0.7)
    if method == 'global':
        axes[1].axvline(threshold, color='red', linewidth=2, label=f'阈值={threshold}')
        axes[1].legend()
    axes[1].set_title('灰度直方图')
    axes[2].imshow(binary, cmap='gray')
    axes[2].set_title(title)
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

interact(
    explore_threshold,
    threshold=IntSlider(value=128, min=0, max=255, step=1, description='阈值'),
    method=Dropdown(value='global', options=['global', 'otsu', 'adaptive'], description='方法'),
    block_size=IntSlider(value=51, min=11, max=199, step=2, description='块大小'),
)
print('请使用上方滑块交互式探索阈值分割参数！')


## 实战案例：细胞计数应用

这是一个完整的分割与计数案例，模拟生物医学图像分析中的细胞计数任务。流程包括：

1. 生成模拟细胞图像（荧光显微镜风格）
2. 预处理（滤波降噪）
3. Otsu 阈值 + 距离变换
4. 分水岭分割分离粘连细胞
5. 计数并计算每个细胞的统计信息（面积、周长等）
6. 可视化标注结果

In [ ]:
# ===== 细胞计数的完整实现 =====

def generate_cell_image(size=512, num_cells=25):
    """生成模拟荧光显微镜细胞图像"""
    np.random.seed(42)
    # 暗背景 + 高斯噪声
    bg = np.random.normal(20, 8, (size, size))
    bg = np.clip(bg, 0, 60)

    image = bg.copy()
    cell_info = []

    attempts = 0
    while len(cell_info) < num_cells and attempts < 300:
        cx = np.random.randint(40, size - 40)
        cy = np.random.randint(40, size - 40)
        # 细胞半径（正态分布模拟大小差异）
        r = int(np.clip(np.random.normal(18, 5), 10, 30))

        # 允许适当重叠（模拟真实组织切片）
        overlap_severe = False
        for (pcx, pcy, pr) in cell_info:
            dist = np.sqrt((cx - pcx)**2 + (cy - pcy)**2)
            if dist < (r + pr) * 0.4:  # 严重重叠则跳过
                overlap_severe = True
                break
        if overlap_severe:
            attempts += 1
            continue

        # 生成细胞（中心亮的椭圆光斑）
        y, x = np.ogrid[:size, :size]
        cell_region = ((x - cx)**2 + (y - cy)**2) <= r**2
        # 高斯亮度分布（中心最亮）
        dist_sq = ((x - cx)**2 + (y - cy)**2) / (r**2)
        intensity = 200 * np.exp(-dist_sq * 2)
        intensity = np.clip(intensity, 0, 220)

        image[cell_region] = np.maximum(image[cell_region], intensity[cell_region])
        cell_info.append((cx, cy, r))
        attempts += 1

    # 添加泊松噪声（模拟光子计数噪声）
    image = np.random.poisson(np.clip(image, 0, 255)).astype(np.float32)
    image = np.clip(image, 0, 255).astype(np.uint8)

    return image, cell_info


# 生成细胞图像
cell_img, true_cells = generate_cell_image(512, 25)
print(f'实际生成的细胞数: {len(true_cells)}')

# ---- 细胞检测与计数 ----
# 1. 高斯滤波降噪
denoised = cv2.GaussianBlur(cell_img, (5, 5), 0)

# 2. Otsu 阈值
_, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 3. 形态学清理
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)
binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

# 4. 距离变换
dist = cv2.distanceTransform(binary, cv2.DIST_L2, 5)

# 5. 寻找种子点
_, sure_fg = cv2.threshold(dist, 0.3 * dist.max(), 255, 0)
sure_fg = np.uint8(sure_fg)

# 6. 分水岭分割
_, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1
sure_bg = cv2.dilate(binary, kernel, iterations=3)
unknown = cv2.subtract(sure_bg, sure_fg)
markers[unknown == 255] = 0
markers = cv2.watershed(cv2.cvtColor(cell_img, cv2.COLOR_GRAY2BGR), markers)

# 7. 统计每个细胞
num_cells = markers.max() - 1
print(f'检测到的细胞数: {num_cells}')

# 计算每个细胞的属性
cell_stats = []
for label_id in range(2, markers.max() + 1):
    cell_mask = (markers == label_id).astype(np.uint8)
    area = np.sum(cell_mask)
    # 查找轮廓计算周长
    contours, _ = cv2.findContours(cell_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        perimeter = cv2.arcLength(contours[0], True)
        # 圆度 = 4*pi*area / perimeter^2 （圆形=1）
        circularity = 4 * np.pi * area / (perimeter * perimeter) if perimeter > 0 else 0
        cy, cx = np.unravel_index(np.argmax(cell_mask * dist), cell_mask.shape)
        cell_stats.append({
            'id': label_id - 1,
            'area': int(area),
            'perimeter': float(perimeter),
            'circularity': float(circularity),
            'center': (cx, cy),
        })

# 打印统计信息
print(f'\n细胞统计信息（前5个）:')
print(f'{"ID":<5} {"面积(px)":<12} {"周长(px)":<12} {"圆度":<10}')
print('-' * 45)
for s in cell_stats[:5]:
    print(f'{s["id"]:<5} {s["area"]:<12} {s["perimeter"]:<12.1f} {s["circularity"]:<10.3f}')
if len(cell_stats) > 5:
    print(f'... 共 {len(cell_stats)} 个细胞')


In [ ]:
# ===== 细胞计数可视化 =====

# 创建标注图像
cell_labeled = cv2.cvtColor(cell_img, cv2.COLOR_GRAY2BGR)

# 为每个细胞画轮廓和编号
for stat in cell_stats:
    label_id = stat['id'] + 1
    cell_mask = (markers == label_id).astype(np.uint8)
    contours, _ = cv2.findContours(cell_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        # 随机颜色
        color = tuple(np.random.randint(50, 255, 3).tolist())
        cv2.drawContours(cell_labeled, contours, -1, color, 2)
        cx, cy = stat['center']
        cv2.putText(cell_labeled, str(stat['id']), (cx - 8, cy + 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1)

# 完整可视化
fig, axes = plt.subplots(2, 3, figsize=(16, 11))

axes[0, 0].imshow(cell_img, cmap='gray')
axes[0, 0].set_title('原始模拟细胞图像\n(荧光显微镜风格)')
axes[0, 0].axis('off')

axes[0, 1].imshow(binary, cmap='gray')
axes[0, 1].set_title('Otsu 阈值 + 形态学清理')
axes[0, 1].axis('off')

axes[0, 2].imshow(dist, cmap='jet')
axes[0, 2].set_title('距离变换')
axes[0, 2].axis('off')

axes[1, 0].imshow(sure_fg, cmap='gray')
axes[1, 0].set_title('确定前景（种子点）')
axes[1, 0].axis('off')

axes[1, 1].imshow(markers, cmap='tab20')
axes[1, 1].set_title('分水岭区域标记')
axes[1, 1].axis('off')

axes[1, 2].imshow(cell_labeled)
axes[1, 2].set_title(f'最终结果\n(检测到 {num_cells} 个细胞)')
axes[1, 2].axis('off')

plt.suptitle('细胞计数应用——完整流程', fontsize=14)
plt.tight_layout()
plt.show()

# 统计汇总
areas = [s['area'] for s in cell_stats]
print(f'\n===== 细胞计数汇总 =====')
print(f'实际细胞数: {len(true_cells)}')
print(f'检测细胞数: {num_cells}')
print(f'面积统计:')
print(f'  最小面积: {min(areas)} px')
print(f'  最大面积: {max(areas)} px')
print(f'  平均面积: {np.mean(areas):.1f} px')
print(f'  面积标准差: {np.std(areas):.1f} px')
print(f'平均圆度: {np.mean([s["circularity"] for s in cell_stats]):.3f}')

## 总结与延伸

### 本模块核心要点

| 知识点 | 核心概念 | 关键函数/类 |
|--------|---------|-----------|
| 阈值分割 | 全局/Otsu/自适应 | threshold_otsu, threshold_local |
| 连通域分析 | 4-连通/8-连通标记 | label, regionprops |
| 分水岭算法 | 距离变换+标记+分割 | watershed, peak_local_max |
| GrabCut | 图割+高斯混合模型 | cv2.grabCut |
| 形态学操作 | 腐蚀/膨胀/开/闭/梯度 | cv2.erode/dilate/morphologyEx |

### 关键技巧

- Otsu 不适合光照不均匀的图像，此时应使用自适应阈值
- 分水岭直接使用会过度分割，必须先做距离变换+标记
- 开运算去白点，闭运算填黑点；椒盐噪声需要先开后闭（或先闭后开）
- 结构元素的形状和大小对形态学结果影响很大

### 延伸阅读

- 《Digital Image Processing》-- Gonzalez & Woods 第 9-10 章
- OpenCV 形态学操作教程
- SLIC 超像素分割（skimage.segmentation.slic）
- U-Net 等深度学习语义分割方法
- 实例分割（Mask R-CNN, YOLACT）与全景分割